# ReServe on one Kaggle GPU

Select a GPU accelerator and enable Internet for the first run. The notebook uses the first visible GPU and the same command as a local install. Save a notebook version with outputs to retain the results archive.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

work = Path("/kaggle/working")
repo = work / "ReServe"
if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "frank-scheduler",
            "https://github.com/DaDanielL/ReServe.git",
            str(repo),
        ],
        check=True,
    )
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "HF_HOME": str(work / "hf-cache")}


def run(*args):
    subprocess.run(list(map(str, args)), cwd=repo, env=env, check=True)


run("git", "rev-parse", "HEAD")
run(sys.executable, "-m", "pip", "install", "-e", ".[live]")
run(
    sys.executable,
    "-c",
    "import torch; assert torch.cuda.is_available(), 'Select a GPU accelerator'; print(torch.__version__, torch.cuda.get_device_name(0))",
)

## Run the comparison

The default uses real JITServe prompts, real BurstGPT arrival gaps scaled to one device, and deadlines learned from this GPU. Set `source = "qwen-math"` to run the published Qwen math prompts. Increase the output cap if many requests reach it. Increase repetitions for reported results. Provide an unused output path for each run.

In [ ]:
source = "jitserve"
output = work / ("reserve-" + source)
settings = [
    "--device",
    "cuda:0",
    "--dtype",
    "float16",
    "--source",
    source,
    "--deadline-mode",
    "device-p95",
    "--calibration",
    "24",
    "--requests",
    "24",
    "--max-prompt-tokens",
    "512",
    "--max-new-tokens",
    "256",
    "--high-kv-tokens",
    "1024",
    "--arrival-scale",
    "0.02",
    "--seed",
    "42",
    "--repeats",
    "1",
    "--data-dir",
    work / "reserve-data",
    "--output",
    output,
]
run(sys.executable, "-m", "recovery_serving.live", *settings)

## Inspect and save

Use the cap count to judge whether deadline attainment is interpretable. The archive contains the frozen workload and budgets, calibration observations, generated token IDs, hardware settings, and per-policy events.

In [ ]:
import json
import shutil

summary = json.loads((output / "summary.json").read_text())
for policy, stats in summary.items():
    print(
        policy,
        {
            k: stats[k]
            for k in (
                "requests",
                "natural_eos",
                "length_capped",
                "deadline_met",
                "deadline_goodput_rps",
                "low_switches",
            )
        },
    )
archive = shutil.make_archive(str(output), "zip", output)
print(archive)